In [1]:
# Install dependencies as needed:
# ! pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "IMDB Dataset.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

/tmp/ipython-input-405851011.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


In [2]:
df.shape

(50000, 2)

In [3]:
df.sample(3)

,review,sentiment
5299,"Well, what can i say about this movie. I'm spe...",negative
16712,I got myself a copy of this film thinking it w...,negative
37682,"I watch a lot of movies - DVD, features, and c...",positive


In [4]:
from sklearn.model_selection import train_test_split

x = df["review"]
y = df["sentiment"]

x_train , x_test , y_train , y_test = train_test_split(x , y , test_size=0.2 , random_state=42)

In [5]:
import spacy
import re


nlp = spacy.load("en_core_web_sm" , disable=["parser" , "ner" , "textcat"])

def preprocess_texts(texts):
    cleaned_texts = [re.sub(r"[^a-zA-Z0-9']+", " ", t).strip() for t in texts]
    
    results = []
    for doc in nlp.pipe(cleaned_texts, batch_size=1000):
        tokens = [token.lemma_.lower() for token in doc if not token.is_punct]
        results.append(" ".join(tokens))
    
    return results


In [6]:
x_train_cleaned = preprocess_texts(x_train.tolist())
x_test_cleaned = preprocess_texts(x_test.tolist())

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=10000)
x_train = vectorizer.fit_transform(x_train_cleaned)
x_test = vectorizer.transform(x_test_cleaned)

In [15]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(x_train , y_train)

LogisticRegression(max_iter=1000)

In [16]:
from sklearn.metrics import classification_report

y_pred = model.predict(x_test)
print(classification_report(y_test , y_pred))


              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      4961
    positive       0.89      0.91      0.90      5039

    accuracy                           0.89     10000
   macro avg       0.90      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [17]:
def predict_sentiment(text):
    cleaned_text = preprocess_texts([text])
    vector = vectorizer.transform(cleaned_text)
    sentiment = model.predict(vector)[0]
    return sentiment

In [21]:
predict_sentiment("im ok")

'negative'

-------------

In [22]:
from sklearn.pipeline import Pipeline


pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("model", model)
])

pipeline.fit(x_train_cleaned , y_train)


Pipeline(steps=[('vectorizer', TfidfVectorizer(max_features=10000)),
                ('model', LogisticRegression(max_iter=1000))])

In [53]:
import spacy
import re


nlp = spacy.load("en_core_web_sm" , disable=["parser" , "ner" , "textcat"])

def preprocess_texts(texts):
    cleaned_texts = [re.sub(r"[^a-zA-Z0-9']+", " ", t).strip() for t in texts]
    
    results = []
    for doc in nlp.pipe(cleaned_texts, batch_size=1000):
        tokens = [token.lemma_.lower() for token in doc if not token.is_punct]
        results.append(" ".join(tokens))
    
    return results



def predict_sentiment(text):
    cleaned_text = preprocess_texts([text])[0]
    return pipeline.predict([cleaned_text])[0]


In [57]:
text = "spider man movie is good"
result = predict_sentiment(text)
result_prob = pipeline.predict_proba([text])[0]
print(f"Result: {result}")
print(f"Probability: {result_prob}")

Result: positive
Probability: [0.30563415 0.69436585]


In [ ]:
# # Function to save and send the zip file that containts the model and the vectorizer

# import joblib
# import shutil
# from google.colab import drive
# import os


# def save_and_send_model(pipeline , file_name):
#     drive.mount('/content/drive')
#     os.makedirs("./assets", exist_ok=True)
#     joblib.dump(pipeline , f"./assets/{file_name}_pipeline.joblib")
#     # joblib.dump(vectorizer , f"./assets/{file_name}_vectorizer.joblib")
#     shutil.make_archive("assets" , "zip" , "./assets")
#     shutil.copy("./assets.zip" , "/content/drive/MyDrive/Colab Notebooks")

In [ ]:
# save_and_send_model(pipeline , "sentiment-analyzer")

Mounted at /content/drive


In [65]:
import os

os.listdir()

['.config', 'assets.zip', 'assets', 'drive', 'sample_data']

In [ ]:
import joblib

pipe = joblib.load("./assets/sentiment-analyzer_pipeline.joblib")

In [68]:

def predict_sentiment(text):
    cleaned_text = preprocess_texts([text])[0]
    return pipe.predict([cleaned_text])[0]



text = "spider man movie is good"
result = predict_sentiment(text)
result_prob = pipe.predict_proba([text])[0]
print(f"Result: {result}")
print(f"Probability: {result_prob}")

Result: positive
Probability: [0.30563415 0.69436585]


In [69]:
pipe.classes_

array(['negative', 'positive'], dtype=object)